In [ ]:
# Colab setup: install fine-tuning dependencies before importing them.
# In Colab, choose Runtime -> Change runtime type -> GPU before running this notebook.
import sys
import subprocess

packages = [
    "transformers>=4.44.0",
    "datasets>=2.20.0",
    "accelerate>=0.33.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.3",
    "sentencepiece",
    "protobuf",
    "huggingface_hub",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])


In [ ]:
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab, choose Runtime -> Change runtime type -> GPU, then rerun.")

print("GPU:", torch.cuda.get_device_name(0))

# Optional Hugging Face login. In Colab, add HF_TOKEN under the key icon on the left if you use a gated model.
hf_token = None
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    from huggingface_hub import login

    login(token=hf_token)

# Make sure the dataset file is available in the Colab runtime.
if not Path("datasets/pascal_alpaca.json").exists():
    try:
        from google.colab import files

        print("Upload datasets/pascal_alpaca.json")
        files.upload()
    except Exception as exc:
        raise FileNotFoundError("datasets/pascal_alpaca.json was not found in the current directory.") from exc

# 1. Configure 4-bit QLoRA quantization.
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# 2. Load an English-focused base model and tokenizer.
# Alternative: meta-llama/Meta-Llama-3.1-8B-Instruct is stronger, but usually requires Hugging Face access approval.
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
hf_kwargs = {"token": hf_token} if hf_token else {}

tokenizer = AutoTokenizer.from_pretrained(model_id, **hf_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    **hf_kwargs,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 3. Configure LoRA adapter parameters.
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 4. Load the Alpaca JSON dataset and convert it to this model's chat-template text format.
dataset = load_dataset("json", data_files="datasets/pascal_alpaca.json", split="train")
INSTRUCTION = dataset[0]["instruction"]

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": f"{example['instruction']}\n\nPlayer: {example['input']}",
        },
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

def tokenize_example(example):
    return tokenizer(example["text"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_example, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Configure training hyperparameters.
training_args = TrainingArguments(
    output_dir="./atai_pascal_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
)

# 6. Fine-tune with the standard Hugging Face Trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()


In [ ]:
# 7. Save the LoRA adapter and tokenizer.
adapter_dir = "./atai_pascal_lora"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved LoRA adapter and tokenizer to {adapter_dir}")

# 8. Run simple tests with the trained adapter.
model.eval()
model.config.use_cache = True

def chat(player_input, max_new_tokens=80):
    messages = [
        {
            "role": "user",
            "content": f"{INSTRUCTION}\n\nPlayer: {player_input}",
        }
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response_ids = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True).strip()

test_prompts = [
    "The clouds looked weird today.",
    "I found a scallop while diving.",
    "I feel like everyone on the island is busy except me.",
    "My snack fell on the floor and I got philosophical about it.",
]

for prompt in test_prompts:
    print("Player:", prompt)
    print("Pascal:", chat(prompt))
    print()

# 9. Optional: merge the LoRA adapter into the base model for deployment. This uses more VRAM/RAM.
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("./atai_pascal_merged", safe_serialization=True)
# tokenizer.save_pretrained("./atai_pascal_merged")
